# Multi-harmonic search statistic 

This notebook generates an EMRI and transforms to time-frequency using short-Fourier-transform (SFT)
We then evaluate the proposed multi-harmonic semi-coherent statistic on the known signal, starting from the true parameters to see how good a job it does at approximating the SNR.

The statistic is built from the two fundamental frequency tracks $f_\phi(t)$ (azimuthal) and $f_r(t)$ (radial). Every harmonic track is

$$ f_{mn}(t) = m\,f_\phi(t) + n\,f_r(t),\qquad \dot f_{mn}(t) = m\,\dot f_\phi(t) + n\,\dot f_r(t), $$

and the multi-harmonic statistic sums the single-harmonic statistic $\Lambda$ over the
harmonic set $\mathcal H=\{(m,n):0\le m\le 10,\;|n|\le 50\}$. This is the same mode content that is in the Kerr Eccentric Equatorial waveform model which is used for injection. The bounds may be tweaked to reduce ciomputational cost. 

$$ \Lambda_{\mathrm{MH}}(f_\phi,f_r) = \sum_{(m,n)\in\mathcal H}
   \Lambda\!\big(\{m f_\phi^{(\alpha)}+n f_r^{(\alpha)},\; m\dot f_\phi^{(\alpha)}+n\dot f_r^{(\alpha)}\}\big). $$

The note `multi-harmonic-search-statistic.md` describes the motivation for this search statistic and details of the construction. This noteboook is purposed to test the implememnatation and shows how it performs in idealized conditions. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

from few.utils.constants import YRSID_SI

from emrisearch.search_utils import generate_emri_signal_and_sfts

# get the single-harmonic search statistic which is the building block of the multi-harmonic search statistic
from emrisearch.jax_utils import det_stat   

np.random.seed(0)

ModuleNotFoundError: No module named 'emrisearch'

## Generate signal and take SFTs

Define the signal parameters as`true_values = [m1, m2, a, T_plunge, e_f, x0]`. 

We pick a moderately eccentric, spinning system and inject it at a reference SNR. `generate_emri_signal_and_sfts` returns the signal/noise/data SFTs and the true two-track frequency evolution evaluated at the SFT mid-times. Note that this is an adapted version of the `generate_emri_signal_and_sfts` function in `emri_search.py` that also returns the true tracks for the two fundemental frequencies.

In [ ]:
true_values = np.asarray([
    1e6, #m1
    10.0,  # m2
    0.95,  # a
    0.5, # e0
    1.7, # Tplunge
    1.0   # xI0  
    ]) 

T_data  = 2.0    # data length [yr]
T_snr   = 2.0     # length used to set the SNR [yr]  (>= T_data)
deltaT  = 5.0     # time step [s]
T_sft   = 5e4     # SFT segment length [s]
snr_ref = 30.0    # injected SNR

inj = generate_emri_signal_and_sfts(true_values, T_data, T_sft, deltaT, snr_ref, T_snr)

# unpack the information from created injection
data_sfts   = jnp.asarray(inj['data_sfts'],   dtype=jnp.complex128)
signal_sfts = jnp.asarray(inj['signal_sfts'], dtype=jnp.complex128)
noise_sfts  = jnp.asarray(inj['noise_sfts'],  dtype=jnp.complex128)
t_alpha     = inj['t_alpha']

# some stats
print(f"SFT grid: {data_sfts.shape[0]} freq bins x {data_sfts.shape[1]} segments")
print(f"Injected SNR: {inj['snr_final']:.1f}")

## The two fundamental tracks 

`inj['true_phi_f_fdot_fddot']` is `(phi, f, fdot, fddot)`, each of shape `(2, n_sft)` where index `0` is the azimuthal ($\phi$) and index `1` the radial ($r$) fundamental frequency. These two $(f,\dot f)$ sequences are the only input the statistic needs and the only two frequencies we are interested in.

In [ ]:
phi, f, fdot, fddot = inj['true_phi_f_fdot_fddot']

f_fund    = jnp.asarray(f,    dtype=jnp.float64)   # (2, n_sft): [f_phi, f_r]
fdot_fund = jnp.asarray(fdot, dtype=jnp.float64)   # (2, n_sft): [fdot_phi, fdot_r]

print("f_phi range [Hz]:", float(jnp.nanmin(f_fund[0])), "->", float(jnp.nanmax(f_fund[0])))
print("f_r   range [Hz]:", float(jnp.nanmin(f_fund[1])), "->", float(jnp.nanmax(f_fund[1])))

## Extended multi-harmonic statistic
This is an extension compared to the work by Speri et al. (2025) where only the dominant azimuthal track was used. The motivaton for this extension is that the multi-mode structure is more pronounced for more extreme mass-ratios and more eccentric systems. Restricting to the dominant mode then risks missing out on a lot of SNR and makes the search statistic less useful for the more interesting systems.  

For each harmonic $(m,n)$ we form the track $m f_\phi + n f_r$ and evaluate the
single-harmonic statistic `det_stat`. We `vmap` over the full harmonic grid and `jit`
the whole thing. The SFTs are computed once; only the cheap integer combinations vary.

In [ ]:
# Harmonic set H = {0<=m<=10, |n|<=50}; (0,0) excluded.
m_grid = np.arange(0, 11)
n_grid = np.arange(-50, 51)
modes = np.array([(m, n) for m in m_grid for n in n_grid if not (m == 0 and n == 0)],
                 dtype=np.float64)
modes = jnp.asarray(modes)
print(f"|H| = {modes.shape[0]} harmonics")

@partial(jax.jit, static_argnums=(4, 5))
def multi_harmonic_per_mode(data_sfts, f_fund, fdot_fund, modes, P, T_sft):
    def one(mn):
        f_a  = mn[0] * f_fund[0]    + mn[1] * f_fund[1]
        fd_a = mn[0] * fdot_fund[0] + mn[1] * fdot_fund[1]
        return det_stat(data_sfts, f_a, fd_a, P=P, T_sft=T_sft)
    return jax.vmap(one)(modes)

def multi_harmonic_stat(data_sfts, f_fund, fdot_fund, modes, P=100, T_sft=5e4):
    per_mode = multi_harmonic_per_mode(data_sfts, f_fund, fdot_fund, modes, P, T_sft)
    return float(per_mode.sum()), np.asarray(per_mode)

## Evaluate on the known signal

We compute the statistic on the signal-only, noise-only and signal+noise SFTs, and compare the single-harmonic $(2,0)$ statistic with the multi-harmonic summed statistic. We compare the relative power we get out of these two statistics compared to the full SNR of the signal, which can be shown to be the optimal search statistic. 

In [ ]:
P = 100
# Single harmonic (2,0)
single = lambda sfts: float(det_stat(sfts, 2*f_fund[0], 2*fdot_fund[0], P=P, T_sft=T_sft))

Lam_single = {k: single(v) for k, v in
              [('signal', signal_sfts), ('noise', noise_sfts), ('data', data_sfts)]}

Lam_multi = {}
per_mode = {}
for k, v in [('signal', signal_sfts), ('noise', noise_sfts), ('data', data_sfts)]:
    Lam_multi[k], per_mode[k] = multi_harmonic_stat(v, f_fund, fdot_fund, modes, P=P, T_sft=T_sft)

print(f"{'':8s}  single (2,0)     multi-harmonic")
for k in ['signal', 'noise', 'data']:
    print(f"{k:8s}  {Lam_single[k]:12.1f}   {Lam_multi[k]:14.1f}")

## Power contribution of different modes

The bar chart shows the per-harmonic contribution $\Lambda^{(m,n)}$ on the signal SFTs: the dominant $(2,0)$ mode plus a tail of eccentricity-induced sidebands $(m,n\neq0)$ that the single-harmonic statistic ignores but the multi-harmonic statistic captures.

In [ ]:
contrib = per_mode['signal']
order = np.argsort(contrib)[::-1]
top = order[:15]
labels = [f"({int(modes[i,0])},{int(modes[i,1])})" for i in top]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(len(top)), contrib[top], color='C0')
ax.set_xticks(range(len(top)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel(r'$\Lambda^{(m,n)}$ (signal)')
ax.set_title('Top per-harmonic contributions to the multi-harmonic statistic')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

frac = contrib[order[0]] / contrib.sum()
print(f"Dominant mode (2,0) carries {frac:.1%} of the signal statistic;")
print(f"the remaining {1-frac:.1%} comes from {np.sum(contrib>1e-3*contrib.max())-1} sub-dominant harmonics.")

## Tracks on spectrogram

The fundamental and a few harmonic tracks overlaid on the spectrogram (this shows the SFT power $|\tilde d_{j,\alpha}|$ per power) confirm that the proposed combinations $m f_\phi + n f_r$ follow the real signal power. This shows the multi-harmonic structure for EMRIs and motivates the use of higher harmonics. 

In [ ]:
# create frequency array for plotting
freqs = np.fft.rfftfreq(inj['samples_per_sft'], deltaT)

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(np.abs(np.asarray(data_sfts)), 
        aspect='auto', 
        origin='lower',
        extent=[t_alpha[0]/86400, t_alpha[-1]/86400, freqs[0], freqs[-1]],
        cmap='viridis')

for (m, n, c, ls) in [(1, 0, 'C3', ':'), 
                      (2, 0, 'r', '-'), 
                      (3, 1, 'C1', '--'), 
                      (2, 1, 'cyan', '-.')]:
    tr = m*np.asarray(f_fund[0]) + n*np.asarray(f_fund[1])
    tr = np.where(tr > 0, tr, np.nan)
    ax.plot(t_alpha/86400, tr, c, ls=ls, lw=1.2, label=f"({m},{n})")

ax.set_ylim(1e-3, 1.5e-2)
ax.set_yscale('log')
ax.set_xlabel('Time [days]')
ax.set_ylabel('Frequency [Hz]')
ax.set_title('Data spectrogram with harmonic tracks for true parameters')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()